**Group members:** Sander Vikse, Amalie Hestad Berntsen 
**GitHub repository:** https://github.com/sirjimcode/inf201-deliverable-4

# Deliverable 4 - INF201 

Submission deadline: 27th Nov 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit Python scripts (.py), a Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebook
    * See Overview and objectives for a list of what your notebook should contain
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together
* We would appriciate it if use English for this assignment, both in the writing and in the video

## Task 1: Temperatures

Create a `Temperature` class that supports Celsius ("C"), Fahrenheit ("F"), and Kelvin ("K"). This class should:
1. Be initialised with a value and a unit (case-insensitive), and raise a `ValueError` if the unit is not one of "C", "F" or "K", or if the temperature is below absolute zero (0 K).
2. Use a tolerance of 1e-9 for floating-point value comparisons
3. Expose `value` and `unit` as read-only attributes
4. Contain the following conversion methods:
    * `to_celsius(self) -> float`:
        * From Kelvin: $C = K - 273.15$
        * From Fahrenheit: $C = (F - 32) \times \frac{5}{9}$
    * `to_fahrenheit(self) -> float`:
        * From Celsius: $F = C \times \frac{9}{5} + 32$
        * From Kelvin: $F = (K - 273.15) \times \frac{9}{5} + 32$
    * `to_kelvin(self) -> float`:
        * From Celsius: $K = C + 273.15$
        * From Fahrenheit: $K = (F - 32) \times \frac{5}{9} + 273.15$
    * `convert_to(self, target_unit: str) -> Temperature`:
        * Returns a new `Temperature` instance converted to `target_unit` (case-insensitive), raising `ValueError` if `target_unit` is invalid.
5. Contain the following string representations:
    * `__repr__(self) -> str`: Returns an unambiguous representation, such as `Temperature(25.0, 'C')`.
    * `__str__(self) -> str`: Returns a human-readable string formatted to two decimal places, such as `25.00 °C`.
6. Contain all six comparisons (==, !=, <, <=, >, >=) so temperatures can be compared across different units (`Temperature(0, "C") == Temperature(32, "F")`). Should return `NotImplemented` if a `Temperature` instance is compared to a non-Temperature instance.

Write a `test_temperature.py` script that imports the `Temperature` class and uses pytest for the following tests:
1. Initialization: Default unit, case-insensitive units, and read-only behavior.
2. Validation: Invalid unit strings and temperatures below absolute zero in "C", "F", and "K".
3. Conversions: Test conversion accuracy across reference points:
    * Absolute zero ($-273.15^\circ\text{C} = -459.67^\circ\text{F} = 0\text{ K}$)
    * Celsius–Fahrenheit crossover ($-40^\circ\text{C} = -40^\circ\text{F} = 233.15\text{ K}$)
    * Freezing point of water ($0^\circ\text{C} = 32^\circ\text{F} = 273.15\text{ K}$)
    * Human body temperature ($37^\circ\text{C} = 98.6^\circ\text{F} = 310.15\text{ K}$)
    * Boiling point of water ($100^\circ\text{C} = 212^\circ\text{F} = 373.15\text{ K}$)
4. Round-trip conversions: Converting "C" -> "F" -> "K" -> "C" reproduces the original Temperature.
5. Cross-unit comparisons: Equality and ordering (<, <=, >, >=) across mixed units.
6. Formatting: Outputs of `str()` and `repr()`

In [ ]:
"""Temperature class supporting Celsius, Fahrenheit and Kelvin (INF201, Task 1)."""

from __future__ import annotations  # allows "-> Temperature" inside the class

import math

_VALID_UNITS = ("C", "F", "K")  # allowed units
_TOLERANCE = 1e-9               # tolerance for comparing floats


def _normalise_unit(unit: str) -> str:
    """Return the unit in upper case, or raise ValueError if it is invalid."""
    if not isinstance(unit, str) or unit.upper() not in _VALID_UNITS:
        raise ValueError(
            f"Invalid unit {unit!r}. Expected one of {', '.join(_VALID_UNITS)}."
        )
    return unit.upper()


class Temperature:
    """A temperature with a value and a unit ("C", "F" or "K")."""

    # __slots__ restricts the object to these two attributes only
    __slots__ = ("_value", "_unit")

    def __init__(self, value: float, unit: str = "C") -> None:
        unit = _normalise_unit(unit)   # validate and normalise the unit ("c" -> "C")
        value = float(value)           # convert to float, e.g. 25 -> 25.0
        self._value = value            # underscore = "private", not meant to be changed
        self._unit = unit
        # Check that the temperature is not below absolute zero (0 K).
        # We use a tolerance because e.g. -459.67 °F may become -0.0000000001 K
        # due to floating-point rounding errors.
        if self.to_kelvin() < -_TOLERANCE:
            raise ValueError(
                f"Temperature {value} {unit} is below absolute zero (0 K)."
            )

    # @property lets us write t.value (without parentheses), but since there
    # is no setter, t.value = 5 raises an AttributeError.
    @property
    def value(self) -> float:
        return self._value

    @property
    def unit(self) -> str:
        return self._unit

    # ----- conversions -----
    def to_celsius(self) -> float:
        if self._unit == "C":
            return self._value
        if self._unit == "K":
            return self._value - 273.15
        return (self._value - 32) * 5 / 9  # from Fahrenheit

    def to_fahrenheit(self) -> float:
        if self._unit == "F":
            return self._value
        if self._unit == "C":
            return self._value * 9 / 5 + 32
        return (self._value - 273.15) * 9 / 5 + 32  # from Kelvin

    def to_kelvin(self) -> float:
        if self._unit == "K":
            return self._value
        if self._unit == "C":
            return self._value + 273.15
        return (self._value - 32) * 5 / 9 + 273.15  # from Fahrenheit

    def convert_to(self, target_unit: str) -> Temperature:
        """Return a NEW Temperature object in the target unit."""
        target_unit = _normalise_unit(target_unit)
        # Dictionary mapping each unit to the matching conversion method
        converters = {
            "C": self.to_celsius,
            "F": self.to_fahrenheit,
            "K": self.to_kelvin,
        }
        new_value = converters[target_unit]()
        # Rounding errors can give e.g. -1e-14 K; set it to 0 instead
        if target_unit == "K" and -_TOLERANCE < new_value < 0:
            new_value = 0.0
        return Temperature(new_value, target_unit)

    # ----- string representations -----
    def __repr__(self) -> str:
        # Unambiguous version for developers, e.g. Temperature(25.0, 'C')
        return f"Temperature({self._value!r}, {self._unit!r})"

    def __str__(self) -> str:
        # Human-readable version with two decimals, e.g. "25.00 °C" or "273.15 K"
        symbol = "K" if self._unit == "K" else f"°{self._unit}"
        return f"{self._value:.2f} {symbol}"

    # ----- comparisons (all done in Kelvin) -----
    def _is_close(self, other: Temperature) -> bool:
        """True if the temperatures are equal within the 1e-9 tolerance."""
        return math.isclose(
            self.to_kelvin(), other.to_kelvin(), rel_tol=0.0, abs_tol=_TOLERANCE
        )

    # All comparisons return NotImplemented if the other object is not a
    # Temperature. Python then tries other options, and in the end
    # == gives False, while < raises a TypeError.
    def __eq__(self, other: object):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self._is_close(other)

    def __ne__(self, other: object):
        if not isinstance(other, Temperature):
            return NotImplemented
        return not self._is_close(other)

    def __lt__(self, other: object):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() < other.to_kelvin() and not self._is_close(other)

    def __le__(self, other: object):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() < other.to_kelvin() or self._is_close(other)

    def __gt__(self, other: object):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() > other.to_kelvin() and not self._is_close(other)

    def __ge__(self, other: object):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() > other.to_kelvin() or self._is_close(other)

    # Equal objects must have equal hashes. Since "equal" here means
    # "almost equal" (tolerance), that is impossible, so the class is unhashable.
    __hash__ = None


In [ ]:
"""Pytest tests for the Temperature class (INF201, Task 1)."""

import pytest
from temperature import Temperature


# 1. Initialisation
def test_default_unit_is_celsius():
    t = Temperature(25)
    assert t.unit == "C"
    assert t.value == 25.0


@pytest.mark.parametrize("unit, expected", [("c", "C"), ("f", "F"), ("k", "K"),
                                            ("C", "C"), ("F", "F"), ("K", "K")])
def test_unit_is_case_insensitive(unit, expected):
    assert Temperature(300, unit).unit == expected


def test_value_is_read_only():
    t = Temperature(10, "C")
    with pytest.raises(AttributeError):
        t.value = 20


def test_unit_is_read_only():
    t = Temperature(10, "C")
    with pytest.raises(AttributeError):
        t.unit = "K"


# 2. Validation
@pytest.mark.parametrize("bad_unit", ["X", "celsius", "", "R", None, 5])
def test_invalid_unit_raises(bad_unit):
    with pytest.raises(ValueError):
        Temperature(0, bad_unit)


@pytest.mark.parametrize("value, unit", [(-273.16, "C"), (-459.68, "F"), (-0.01, "K"),
                                         (-1000, "C"), (-500, "F"), (-1, "K")])
def test_below_absolute_zero_raises(value, unit):
    with pytest.raises(ValueError):
        Temperature(value, unit)


@pytest.mark.parametrize("value, unit", [(-273.15, "C"), (-459.67, "F"), (0, "K")])
def test_exactly_absolute_zero_is_allowed(value, unit):
    Temperature(value, unit)  # should not raise


@pytest.mark.parametrize("bad_target", ["X", "kelvin", ""])
def test_convert_to_invalid_unit_raises(bad_target):
    with pytest.raises(ValueError):
        Temperature(0, "C").convert_to(bad_target)


# 3. Conversions at reference points
REFERENCE_POINTS = [
    # (celsius, fahrenheit, kelvin)
    (-273.15, -459.67, 0.0),     # absolute zero
    (-40.0, -40.0, 233.15),      # C/F crossover
    (0.0, 32.0, 273.15),         # freezing point of water
    (37.0, 98.6, 310.15),        # human body temperature
    (100.0, 212.0, 373.15),      # boiling point of water
]


@pytest.mark.parametrize("c, f, k", REFERENCE_POINTS)
def test_conversions_from_celsius(c, f, k):
    t = Temperature(c, "C")
    assert t.to_celsius() == pytest.approx(c, abs=1e-9)
    assert t.to_fahrenheit() == pytest.approx(f, abs=1e-9)
    assert t.to_kelvin() == pytest.approx(k, abs=1e-9)


@pytest.mark.parametrize("c, f, k", REFERENCE_POINTS)
def test_conversions_from_fahrenheit(c, f, k):
    t = Temperature(f, "F")
    assert t.to_celsius() == pytest.approx(c, abs=1e-9)
    assert t.to_fahrenheit() == pytest.approx(f, abs=1e-9)
    assert t.to_kelvin() == pytest.approx(k, abs=1e-9)


@pytest.mark.parametrize("c, f, k", REFERENCE_POINTS)
def test_conversions_from_kelvin(c, f, k):
    t = Temperature(k, "K")
    assert t.to_celsius() == pytest.approx(c, abs=1e-9)
    assert t.to_fahrenheit() == pytest.approx(f, abs=1e-9)
    assert t.to_kelvin() == pytest.approx(k, abs=1e-9)


@pytest.mark.parametrize("c, f, k", REFERENCE_POINTS)
def test_convert_to_returns_new_temperature(c, f, k):
    t = Temperature(c, "C")
    tf = t.convert_to("f")
    tk = t.convert_to("K")
    assert isinstance(tf, Temperature) and tf.unit == "F"
    assert isinstance(tk, Temperature) and tk.unit == "K"
    assert tf.value == pytest.approx(f, abs=1e-9)
    assert tk.value == pytest.approx(k, abs=1e-9)
    assert t.unit == "C"  # original unchanged


# 4. Round-trip conversions
@pytest.mark.parametrize("c", [-273.15, -40.0, 0.0, 25.0, 37.0, 100.0, 1234.5678])
def test_round_trip_c_f_k_c(c):
    original = Temperature(c, "C")
    result = original.convert_to("F").convert_to("K").convert_to("C")
    assert result.unit == "C"
    assert result == original
    assert result.value == pytest.approx(c, abs=1e-9)


# 5. Cross-unit comparisons
@pytest.mark.parametrize("c, f, k", REFERENCE_POINTS)
def test_equality_across_units(c, f, k):
    tc, tf, tk = Temperature(c, "C"), Temperature(f, "F"), Temperature(k, "K")
    assert tc == tf and tf == tk and tc == tk
    assert not (tc != tf)
    assert tc <= tf and tc >= tk
    assert not (tc < tf) and not (tc > tk)


def test_inequality_across_units():
    assert Temperature(0, "C") != Temperature(0, "F")
    assert Temperature(0, "K") != Temperature(0, "C")


def test_ordering_across_units():
    cold = Temperature(0, "C")         # 273.15 K
    warm = Temperature(98.6, "F")      # 310.15 K
    hot = Temperature(373.15, "K")     # 100 C
    assert cold < warm < hot
    assert hot > warm > cold
    assert cold <= warm <= hot
    assert hot >= warm >= cold
    assert not (hot < cold)
    assert not (cold >= warm)


def test_equality_within_tolerance():
    assert Temperature(0, "C") == Temperature(0 + 1e-10, "C")
    assert Temperature(0, "C") != Temperature(0 + 1e-6, "C")


def test_comparison_with_non_temperature_returns_notimplemented():
    t = Temperature(0, "C")
    for method in ("__eq__", "__ne__", "__lt__", "__le__", "__gt__", "__ge__"):
        assert getattr(t, method)(0) is NotImplemented
    assert (t == 0) is False
    assert (t != "0 C") is True
    with pytest.raises(TypeError):
        t < 5


def test_sorting_mixed_units():
    temps = [Temperature(212, "F"), Temperature(0, "K"), Temperature(20, "C")]
    assert [t.unit for t in sorted(temps)] == ["K", "C", "F"]


# 6. Formatting
@pytest.mark.parametrize("t, expected", [
    (Temperature(25, "C"), "25.00 °C"),
    (Temperature(98.6, "F"), "98.60 °F"),
    (Temperature(273.15, "K"), "273.15 K"),
    (Temperature(-40, "c"), "-40.00 °C"),
    (Temperature(1 / 3, "C"), "0.33 °C"),
])
def test_str(t, expected):
    assert str(t) == expected


@pytest.mark.parametrize("t, expected", [
    (Temperature(25, "C"), "Temperature(25.0, 'C')"),
    (Temperature(98.6, "f"), "Temperature(98.6, 'F')"),
    (Temperature(0, "K"), "Temperature(0.0, 'K')"),
])
def test_repr(t, expected):
    assert repr(t) == expected


def test_repr_round_trips_with_eval():
    t = Temperature(36.6, "C")
    assert eval(repr(t)) == t


In [ ]:
!pytest test_temperature.py -v

## Task 2: Fractions

Create a `Fraction` class representing fractions. Both numerator and denominator should be integers, and the denominator should be stricktly positive (>0). The class should implement the following special methods (aka magic methods, dunder methods): 
* initialiser
* all comparisons (equal, not equal, smaller than, smaller than or equal, greater than, and greated than or equal)
* at least four other special methods

Write a test script to be used with `pytest` for testing the implemented methods. Include the output from testing in a Jupyter notebok.

In [ ]:
"""Fraction class (INF201, Task 2)."""


class Fraction:
    """Fraction with integer parts and a positive denominator."""

    def __init__(self, numerator, denominator):
        # Both parts must be integers
        if not isinstance(numerator, int) or not isinstance(denominator, int):
            raise TypeError("numerator and denominator must be integers")
        if denominator == 0:
            raise ZeroDivisionError("denominator cannot be zero")
        # Move the sign to the numerator: 1/-2 -> -1/2
        if denominator < 0:
            numerator = -numerator
            denominator = -denominator
        self.numerator = numerator
        self.denominator = denominator

    # Comparisons
    # Cross-multiply: a/b vs c/d  ->  a*d vs c*b
    def __eq__(self, other):  # ==
        return self.numerator * other.denominator == other.numerator * self.denominator

    def __ne__(self, other):  # !=
        return self.numerator * other.denominator != other.numerator * self.denominator

    def __lt__(self, other):  # <
        return self.numerator * other.denominator < other.numerator * self.denominator

    def __le__(self, other):  # <=
        return self.numerator * other.denominator <= other.numerator * self.denominator

    def __gt__(self, other):  # >
        return self.numerator * other.denominator > other.numerator * self.denominator

    def __ge__(self, other):  # >=
        return self.numerator * other.denominator >= other.numerator * self.denominator

    # Arithmetic
    def __add__(self, other):
        # a/b + c/d = (a*d + c*b) / (b*d)
        return Fraction(self.numerator * other.denominator + other.numerator * self.denominator,
                        self.denominator * other.denominator)

    def __sub__(self, other):
        # a/b - c/d = (a*d - c*b) / (b*d)
        return Fraction(self.numerator * other.denominator - other.numerator * self.denominator,
                        self.denominator * other.denominator)

    def __mul__(self, other):
        # a/b * c/d = (a*c) / (b*d)
        return Fraction(self.numerator * other.numerator,
                        self.denominator * other.denominator)

    def __truediv__(self, other):
        # a/b / c/d = (a*d) / (b*c)
        return Fraction(self.numerator * other.denominator,
                        self.denominator * other.numerator)


In [ ]:
"""Pytest tests for the Fraction class (INF201, Task 2)."""

import pytest
from fraction import Fraction


# Initialiser
def test_init():
    f = Fraction(3, 4)
    assert f.numerator == 3
    assert f.denominator == 4


def test_negative_denominator_becomes_positive():
    # 1/-2 should be stored as -1/2
    f = Fraction(1, -2)
    assert f.numerator == -1
    assert f.denominator == 2


def test_zero_denominator_raises():
    with pytest.raises(ZeroDivisionError):
        Fraction(1, 0)


def test_non_integer_raises():
    # Floats are not allowed
    with pytest.raises(TypeError):
        Fraction(1.5, 2)
    with pytest.raises(TypeError):
        Fraction(1, 2.0)


# Comparisons
def test_eq():
    # 1/2 and 2/4 are equal
    assert Fraction(1, 2) == Fraction(2, 4)
    assert not Fraction(1, 2) == Fraction(1, 3)


def test_ne():
    assert Fraction(1, 2) != Fraction(1, 3)
    assert not Fraction(1, 2) != Fraction(2, 4)


def test_lt():
    assert Fraction(1, 3) < Fraction(1, 2)
    assert not Fraction(1, 2) < Fraction(1, 2)


def test_le():
    assert Fraction(1, 3) <= Fraction(1, 2)
    assert Fraction(1, 2) <= Fraction(2, 4)
    assert not Fraction(2, 3) <= Fraction(1, 2)


def test_gt():
    assert Fraction(3, 4) > Fraction(2, 3)
    assert not Fraction(1, 2) > Fraction(1, 2)


def test_ge():
    assert Fraction(3, 4) >= Fraction(2, 3)
    assert Fraction(2, 4) >= Fraction(1, 2)
    assert not Fraction(1, 3) >= Fraction(1, 2)


def test_comparison_with_negative_fractions():
    # Sign position should not matter
    assert Fraction(-1, 2) < Fraction(1, 3)
    assert Fraction(1, -2) == Fraction(-1, 2)


# Arithmetic
def test_add():
    # 1/2 + 1/3 = 5/6
    assert Fraction(1, 2) + Fraction(1, 3) == Fraction(5, 6)


def test_sub():
    # 1/2 - 1/3 = 1/6
    assert Fraction(1, 2) - Fraction(1, 3) == Fraction(1, 6)


def test_mul():
    # 2/3 * 3/4 = 1/2
    assert Fraction(2, 3) * Fraction(3, 4) == Fraction(1, 2)


def test_truediv():
    # 1/2 / 1/4 = 2
    assert Fraction(1, 2) / Fraction(1, 4) == Fraction(2, 1)


def test_division_by_negative_fraction_keeps_denominator_positive():
    # 1/2 / -1/3 = -3/2
    result = Fraction(1, 2) / Fraction(-1, 3)
    assert result == Fraction(-3, 2)
    assert result.denominator > 0


def test_division_by_zero_fraction_raises():
    with pytest.raises(ZeroDivisionError):
        Fraction(1, 2) / Fraction(0, 1)


In [ ]:
!pytest test_fraction.py -v